#### Difficulty-targeted Online Data Selection

Here are the specific hyperparameters the authors used for their experimental setup, divided into general reinforcement learning settings and the specific settings for their DOTS and RR algorithms.

**General RL Fine-Tuning & Optimizer Settings**
*   **Optimizer:** AdamW
*   **Learning Rate:** 1e-6 (with a constant schedule)
*   **Weight Decay & Warm-up Ratio:** 0
*   **Total Batch Size:** 512 
*   **Mini Batch Size:** 64
*   **Number of Training Steps:** 60
*   **Number of Gradient Steps:** 480 (derived from 8 gradient steps per training step)
*   **GPU Hardware:** 8x NVIDIA L40S (for 1.5B and 3B models) or 8x NVIDIA A100 (for the 7B model)

**Generation & Rollout Settings**
*   **Max Prompt Length:** 1024 tokens
*   **Max Rollout Length:** 3072 tokens (for Qwen2.5-Math series models) or 4096 tokens (for Qwen2.5-3B)
*   **Number of Rollouts Per Prompt:** 8
*   **Rollout Sampling Temperature:** 0.6
*   **Rollout Sampling Top-p:** 0.95

**DOTS and RR Specific Hyperparameters**
*   **Target Difficulty ($\alpha$):** 0.5 (The ideal predicted failure rate they target for training batch selection).
*   **Reference Set Size ($K$):** 256 (The number of questions randomly sampled at each selection step to compute exact ground-truth difficulty).
*   **Data Sampling Temperature ($\tau$):** 1e-3 (Used to control the sharpness of the probability distribution when selecting questions based on difficulty).
*   **Fresh Rollout Fraction ($\delta$):** 0.5 (The percentage of the total batch size that generates brand new answers).
*   **Buffer Capacity ($C$):** 256 or 512 (The maximum number of recent, informative rollouts stored in the FIFO replay buffer).

In [21]:
from datasets import load_dataset

ds = load_dataset("trl-lib/DeepMath-103K")

def format_prompt(example):
    # The prompt is a list of messages. We modify the content of the user message.
    # We append the formatting instruction to the end of the existing question.
    instruction = "\nLet's think step by step and output the final answer after \"####\"."
    
    # example['prompt'] is a list: [{'role': 'user', 'content': '...'}]
    example['prompt'][0]['content'] += instruction
    return example

# Apply the transformation
dataset = ds.map(format_prompt)

# Verify the first example
print(dataset['train'][0]['prompt'][0]['content'])

Is it possible to construct an uncountable set of subsets of the positive integers, \( T \), such that for any two subsets \( C, D \) in \( T \), either \( C \) is a subset of \( D \) or \( D \) is a subset of \( C \)? Provide a justification for your answer.
Let's think step by step and output the final answer after "####".


In [22]:
dataset['train'][0]

{'prompt': [{'content': 'Is it possible to construct an uncountable set of subsets of the positive integers, \\( T \\), such that for any two subsets \\( C, D \\) in \\( T \\), either \\( C \\) is a subset of \\( D \\) or \\( D \\) is a subset of \\( C \\)? Provide a justification for your answer.\nLet\'s think step by step and output the final answer after "####".',
   'role': 'user'}],
 'solution': 'Yes'}

In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

def RollOut(prompt:str) -> str:
    # Load model directly
    messages = [
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])
    print(response)

    return response

In [24]:
response = RollOut(dataset['train'][0]['prompt'][0]['content'])

To prove this statement, we need to show that it is impossible to construct an uncountable set \( T \) of subsets of the positive integers \( N = \{1, 2, ..., n\} \) such that for any two subsets \( C, D \) in \( T \), either \( C \) is a subset of \( D \) or \( D \) is a subset of \( C \).

Consider the union of all these sets \( T \). For each \( C \in T \), there exists at least one \( D \in T \) such that \( C \subseteq D \). Thus, every element of \( T \) can be uniquely identified with its complement \( T - C \).

Now, consider the set \( S = \{C \in T: C \cap T = \emptyset\} \). Since \( T \) is uncountable, \( S \) must contain more than one element. Therefore, there exist \( C_1, C_2, ..., C_n \) such that \( C_i \cap T = \emptyset \) for some \( i \geq 1 \).

For any two distinct \( C_i, C_j \) in \( S \), since \( C_i \cap T = \emptyset \) and \( C_j \cap T = \emptyset \), it follows that \( C_i \neq C_j \). This implies that \( C_i \) and \( C_j \) cannot both belong to \( 

In [16]:
import re
match = re.search(r"####?\s*(.*)", response)
if match:
    predicted = match.group(1).strip()
    print("Predicted:", predicted)

Predicted: Induction Basis


#### DOTS

In [ ]:
# Hyperparameters
K = 128 # Reference data size

In [ ]:
# Stage 1: Sampling a small "reference set"

reference_dataset = ds['train'].shuffle().select(range(256))
reference_dataset[0]

In [ ]:
reference_dataset[0]['prompt'][0]['content']

In [ ]:
reference_dataset[0]['solution']

In [ ]:
RollOut(reference_dataset[0]['prompt'][0]['content'])

rl_training/verl/docs/preparation/prepare_data.rst

## Convert Dataset to Embeddings

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Math-1.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Math-1.5B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch


#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


# Sentences we want sentence embeddings for
sentences = ["In this paper we show how to compute the $\\Lambda_{\\alpha}$ norm, $\\alpha\\ge 0$, using the dyadic grid. This result is a consequence of the description of the Hardy spaces $H^p(R^N)$ in terms of dyadic and special atoms.",
             "We show that a determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata. The proof involves a bijection from these automata to certain marked lattice paths and a sign-reversing involution to evaluate the determinant."]

# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained('math-similarity/Bert-MLM_arXiv-MP-class_zbMath')
model = AutoModel.from_pretrained('math-similarity/Bert-MLM_arXiv-MP-class_zbMath')

# Tokenize sentences
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling. In this case, mean pooling.
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

print("Sentence embeddings:")
print(sentence_embeddings)


#### GRPO 

In [ ]:
def reward_fn(generated:str, actual:str) -> bool:

    return 1 if generated==actual else 0

In [ ]:
# train_grpo.py
from datasets import load_dataset
from trl import GRPOTrainer
from trl.rewards import accuracy_reward

dataset = load_dataset("trl-lib/DeepMath-103K", split="train")

trainer = GRPOTrainer(
    model="Qwen/Qwen2-0.5B-Instruct",
    reward_funcs=accuracy_reward,
    train_dataset=dataset,
)
trainer.train()